# EDA - Fake News Detection (Agent 1)
Exploration only. No cleaning, no training. Original files are never modified.

In [1]:
from pathlib import Path
import pandas as pd

ROOT = Path.cwd()
while not (ROOT / "requirements.txt").exists():
    ROOT = ROOT.parent
DATA_DIR = ROOT / "data" / "raw" / "fake_news"

fake = pd.read_csv(DATA_DIR / "fake.csv")
true = pd.read_csv(DATA_DIR / "true.csv")
print("fake :", fake.shape, "| true :", true.shape)

fake : (23481, 4) | true : (21417, 4)


In [2]:
print(fake.dtypes.to_string())
print()
print(true.dtypes.to_string())

title      str
text       str
subject    str
date       str

title      str
text       str
subject    str
date       str


In [3]:
print(fake.isna().sum().to_string())
print()
print(true.isna().sum().to_string())

title      0
text       0
subject    0
date       0

title      0
text       0
subject    0
date       0


In [4]:
for name, df in [("fake", fake), ("true", true)]:
    print(name, "| lignes identiques :", df.duplicated().sum(),
          "| titres en double :", df["title"].duplicated().sum())

fake | lignes identiques : 3 | titres en double : 5578
true | lignes identiques : 206 | titres en double : 591


In [5]:
counts = pd.Series({"fake (label 1)": len(fake), "true (label 0)": len(true)})
print(counts.to_string())
print((counts / counts.sum() * 100).round(1).to_string()) 

fake (label 1)    23481
true (label 0)    21417
fake (label 1)    52.3
true (label 0)    47.7


In [6]:
print(fake["subject"].value_counts().to_string())
print()
print(true["subject"].value_counts().to_string())

subject
News               9050
politics           6841
left-news          4459
Government News    1570
US_News             783
Middle-east         778

subject
politicsNews    11272
worldnews       10145


In [7]:
for name, df in [("fake", fake), ("true", true)]:
    dates = pd.to_datetime(df["date"], errors="coerce", format="mixed")
    print(name, "| dates invalides :", dates.isna().sum(),
          "| de", dates.min().date(), "à", dates.max().date())

fake | dates invalides : 10 | de 2015-03-31 à 2018-02-19
true | dates invalides : 0 | de 2016-01-13 à 2017-12-31


In [8]:
for name, df in [("fake", fake), ("true", true)]:
    words = df["text"].fillna("").str.split().str.len()
    print(name, "(nombre de mots par texte)")
    print(words.describe().round(1).to_string())
    print("textes vides :", (words == 0).sum(), "| moins de 10 mots :", (words < 10).sum())
    print()

fake (nombre de mots par texte)
count    23481.0
mean       423.2
std        408.4
min          0.0
25%        240.0
50%        363.0
75%        506.0
max       8135.0
textes vides : 630 | moins de 10 mots : 866

true (nombre de mots par texte)
count    21417.0
mean       385.6
std        274.0
min          0.0
25%        148.0
50%        359.0
75%        525.0
max       5172.0
textes vides : 1 | moins de 10 mots : 1



In [9]:
import re
from collections import Counter
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

def top_words(series, n=15):
    counter = Counter()
    for text in series.fillna(""):
        counter.update(w for w in re.findall(r"[a-z]+", text.lower())
                       if w not in ENGLISH_STOP_WORDS)
    return counter.most_common(n)

print("fake :", top_words(fake["text"]))
print()
print("true :", top_words(true["text"]))

fake : [('s', 142571), ('trump', 79939), ('t', 46282), ('said', 33763), ('president', 27811), ('people', 26591), ('just', 20570), ('clinton', 19213), ('obama', 18838), ('like', 18117), ('donald', 17790), ('news', 14688), ('new', 14416), ('hillary', 14187), ('time', 13880)]

true : [('s', 142720), ('said', 99062), ('trump', 54702), ('u', 44596), ('reuters', 28976), ('president', 28179), ('state', 21025), ('government', 18846), ('new', 16917), ('states', 16653), ('house', 16640), ('republican', 16243), ('united', 15590), ('people', 15287), ('year', 14777)]


In [10]:
for name, df in [("fake", fake), ("true", true)]:
    share = df["text"].fillna("").str.contains("Reuters").mean() * 100
    print(f"{name} : {share:.1f} % des textes contiennent 'Reuters'")

fake : 1.3 % des textes contiennent 'Reuters'
true : 99.8 % des textes contiennent 'Reuters'


## Suitability for a Naive Bayes classifier
- Text column to use: `text` (optionally `title`).
- Label: none in the files; derived from the file (fake / true).
- Classes: reasonably balanced (about 52% / 48%).
- Issues found: duplicates, empty texts, a source signature ("Reuters") almost only in true articles, `subject` and `date` not reliable.
- Conclusion: suitable for a first Naive Bayes baseline, with the limits documented in docs/datasets/fake-news.md.